# Large Language Models

## Introducción a los Modelos de Lenguaje Grande (LLMs)

Los **Modelos de Lenguaje Grande** o **LLMs** son modelos neuronales entrenados sobre grandes colecciones de texto para aprender patrones lingüísticos y generar, transformar o analizar lenguaje natural. En los LLMs autoregresivos, el objetivo básico consiste en **predecir el siguiente token** a partir del contexto anterior. Este enfoque se conoce como **causal language modeling** y es la base de muchos sistemas modernos de generación de texto. [Fuente oficial de Hugging Face sobre causal language modeling](https://huggingface.co/docs/transformers/tasks/language_modeling).

Aunque durante mucho tiempo se han asociado solo con la generación de texto, en la práctica los LLMs actuales pueden utilizarse para muchas tareas: clasificación, resumen, traducción, extracción de información, pregunta-respuesta, asistentes conversacionales, generación de código e incluso flujos multimodales cuando el modelo acepta texto, imagen, audio u otras entradas.

## Componentes de un LLM

#### Tokenización y embeddings
Antes de procesar un texto, el modelo lo divide en **tokens**. Un token no tiene por qué coincidir con una palabra completa: puede ser una palabra, una parte de una palabra, un signo de puntuación o incluso un fragmento frecuente. Después, cada token se transforma en un vector numérico mediante una **capa de embeddings**. Esos vectores son la representación matemática que el modelo utiliza para operar.

En la práctica, muchos modelos trabajan con subpalabras porque eso reduce el tamaño del vocabulario y permite manejar palabras raras o desconocidas. Por ejemplo, una palabra como *ingeniería* podría dividirse en varios sub-tokens según el tokenizador empleado.

#### Información posicional
Como la autoatención no procesa la secuencia de manera recurrente, el modelo necesita saber **en qué posición aparece cada token**. Para ello se añade información posicional, ya sea mediante embeddings posicionales aprendidos o variantes más modernas según la arquitectura concreta.

#### Autoatención
El mecanismo de **self-attention** permite que cada token calcule qué importancia deben tener los demás tokens de la secuencia al construir su representación contextual. Gracias a ello, el modelo puede capturar dependencias de largo alcance sin recorrer la frase palabra a palabra como ocurría en arquitecturas recurrentes.

Por ejemplo, en una oración como *"El ingeniero revisó el código que había producido anteriormente"*, el modelo puede relacionar *ingeniero* con *producido* si el contexto semántico lo requiere.

#### Multi-Head Self-Attention
La **multi-head self-attention** ejecuta varias atenciones en paralelo. Cada cabeza puede centrarse en distintos tipos de relaciones: concordancia, dependencia sintáctica, contexto semántico, referencias internas, etc. Después, sus salidas se combinan y pasan por capas adicionales.

#### Bloques Transformer
Un bloque Transformer típico incluye:
- autoatención,
- conexiones residuales,
- normalización,
- y una red feed-forward.

Apilando muchos bloques, el modelo construye representaciones cada vez más abstractas del texto.

#### Entrenamiento y adaptación
Un LLM suele pasar por varias fases:
1. **Preentrenamiento**: aprende patrones generales del lenguaje a gran escala.
2. **Ajuste fino (fine-tuning / instruction tuning)**: se adapta a tareas o comportamientos concretos.
3. **Uso en inferencia**: genera respuestas para nuevas entradas.

### Ejemplo de Código en Python

Para este ejemplo, utilizaremos la biblioteca `transformers` de Hugging Face. En versiones actuales, para generación de texto suele recomendarse trabajar con `AutoTokenizer` y `AutoModelForCausalLM`, porque son clases genéricas que se adaptan mejor a diferentes checkpoints. [Documentación oficial de pipelines y clases auto](https://huggingface.co/docs/transformers/main_classes/pipelines).

1. **Instalación de la biblioteca `transformers`**: en el entorno de trabajo debe instalarse `transformers` junto con `torch` antes de ejecutar las celdas de generación.

2. **Código para descargar y probar un modelo causal pequeño**: el desarrollo práctico aparece en las **dos celdas de código siguientes**, donde primero se cargan el tokenizador y el modelo, y después se define una función de generación con distintos *prompts* de prueba.

### Explicación del Código

1. **Uso de clases genéricas**: `AutoTokenizer` y `AutoModelForCausalLM` permiten cargar modelos compatibles con generación causal de una forma más flexible.
2. **Modelo de ejemplo**: `distilgpt2` es una versión reducida de GPT-2, útil para aprendizaje y pruebas rápidas.
3. **Generación**: `model.generate(...)` crea la salida token a token. Parámetros como `temperature` y `top_p` controlan la diversidad de la respuesta.
4. **`max_new_tokens`**: hoy se usa con frecuencia en lugar de `max_length`, porque expresa con más claridad cuántos tokens nuevos puede generar el modelo.
5. **`pad_token_id`**: se fija para evitar advertencias y gestionar correctamente la generación.

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 7345.37it/s]


In [7]:
def generate_text(prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text

prompts = [
    "Once upon a time",
    "In a galaxy far, far away",
    "The future of AI is",
    "El futuro de la inteligencia artificial es"
]

for prompt in prompts:
    generated_text = generate_text(prompt)
    print(f"Prompt: {prompt}\nGenerated Text: {generated_text}\n")

Prompt: Once upon a time
Generated Text: Once upon a time of great pain in my stomach, I was given a medicine that I couldn't even touch. In the midst of my long ordeal, I fell into a sleep. I was afraid of dying and had no hope to stop my suffering. I was told to go to bed by a friend to ensure I

Prompt: In a galaxy far, far away
Generated Text: In a galaxy far, far away, far away, far away. A galaxy far away from the galaxy will not only be very distant, but will also be an extremely exciting place to explore.





























Prompt: The future of AI is
Generated Text: The future of AI is not a question of the future of machines, but rather the future of the human race. Humans have the capacity to act independently of all of us and to solve our problems and solve them ourselves.


A study by the University of Oxford (Oxford University Press, 2014) was published in

Prompt: El futuro de la inteligencia artificial es
Generated Text: El futuro de la inteligencia artificial

#### Modelos preentrenados
La librería Transformers de Hugging Face ofrece una gran variedad de modelos preentrenados para tareas de NLP, visión y multimodalidad. Los **pipelines** proporcionan una interfaz de alto nivel y simplifican la inferencia para tareas como clasificación, NER, generación, pregunta-respuesta o resumen. Según la documentación oficial, `pipeline()` abstrae gran parte del preprocesado y de la inferencia y sirve como una vía sencilla de uso para múltiples tareas. [Documentación oficial](https://huggingface.co/docs/transformers/main_classes/pipelines).

La elección del modelo depende de la tarea:
- **Clasificación de texto**: modelos encoder como BERT, RoBERTa o variantes especializadas.
- **Generación de texto**: modelos causales como GPT-2, DistilGPT2, Gemma, Llama, Qwen u otros compatibles.
- **Masked language modeling**: modelos como BERT o RoBERTa.
- **Pregunta-respuesta extractiva**: modelos ajustados sobre datasets como SQuAD.

El uso de modelos preentrenados ahorra tiempo y recursos porque reutiliza conocimiento ya aprendido. Después, si es necesario, puede aplicarse **fine-tuning** sobre un conjunto de datos específico.

#### Pipelines
Hugging Face proporciona pipelines como interfaces de alto nivel para realizar tareas sin manipular manualmente toda la tokenización o las salidas del modelo. Entre los pipelines más habituales se encuentran:
- **Text generation**: `pipeline("text-generation")`
- **Text classification**: `pipeline("text-classification")`
- **Named entity recognition**: `pipeline("ner")`
- **Question answering**: `pipeline("question-answering")`
- **Fill-mask**: `pipeline("fill-mask")`
- **Summarization** y **translation**

Cuando se necesita un mayor control, es preferible trabajar directamente con `AutoTokenizer`, `AutoModel`, `AutoModelForCausalLM` o las clases específicas del problema. Esto permite ajustar parámetros de generación, batching, dispositivos y formatos de salida con más precisión.

### Resumen de Casos de Uso de Transformers

#### 1. Clasificación de Texto
La clasificación de texto es una de las aplicaciones más comunes del NLP: análisis de sentimientos, detección de spam, clasificación temática, moderación de contenido, etc. Utilizando modelos preentrenados y pipelines, se pueden asignar etiquetas de forma muy sencilla.

> Nota: la etiqueta devuelta depende del modelo concreto y del tipo de texto para el que fue entrenado. Por eso conviene revisar siempre la ficha del modelo antes de interpretar las clases.

#### 2. Pregunta-Respuesta
En esta tarea se proporciona un **contexto** y una **pregunta**, y el modelo trata de extraer la respuesta más probable del texto.

El ejemplo práctico correspondiente aparece en la **celda de código siguiente**, donde se utiliza un pipeline de *question answering* con un modelo ajustado sobre SQuAD2 para extraer una respuesta directamente desde el contexto dado.

#### 3. Fine-Tuning
El **ajuste fino** permite adaptar un modelo preentrenado a una tarea concreta con datos propios. En la práctica, hoy es habitual trabajar con clases `Auto...` y con `Trainer` cuando se quiere un flujo de entrenamiento sencillo y reproducible.

Un flujo típico de fine-tuning incluiría estas fases:
1. Preparar un conjunto de entrenamiento con textos y etiquetas.
2. Tokenizar las entradas con el tokenizador del modelo base.
3. Construir un dataset que devuelva tensores compatibles con PyTorch.
4. Cargar un modelo de clasificación con el número de clases adecuado.
5. Definir los argumentos de entrenamiento.
6. Entrenar y posteriormente evaluar sobre validación o test.

Para realizar inferencia con el modelo ajustado, el procedimiento general consiste en:
- tokenizar un nuevo texto,
- ejecutar el modelo en modo evaluación,
- obtener los *logits*,
- y escoger la clase más probable.

En un entorno real, este proceso debería incluir partición de datos, validación, métricas, control de sobreajuste y, si procede, ajuste de hiperparámetros.

In [3]:
from transformers import pipeline

classifier = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-sentiment-latest")
input_text = "Este curso de Ingeniería de Software es fantástico."
prediction = classifier(input_text)
print(prediction)
# Output: [{'label': 'neutral', 'score': 0.7426337003707886}]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 49023.38it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'label': 'neutral', 'score': 0.7426338791847229}]


In [8]:
from transformers import pipeline

qa_pipe = pipeline("question-answering", model="deepset/roberta-base-squad2")
contexto = """
La arquitectura Transformer se basa en mecanismos de self-attention para modelar la dependencia
entre tokens de la secuencia. Fue introducida en 2017 por Vaswani et al. en el artículo
'Attention Is All You Need'.
"""
pregunta = "¿Qué mecanismo emplea la arquitectura Transformer para modelar la dependencia entre tokens?"
resultado = qa_pipe(question=pregunta, context=contexto)
print("Pregunta:", pregunta)
print("Respuesta:", resultado['answer'])
print("Probabilidad:", resultado['score'])
# Output: Pregunta: ¿Qué mecanismo emplea la arquitectura Transformer para modelar la dependencia entre tokens?
#         Respuesta: self-attention
#         Probabilidad: 0.7234193682670593

KeyError: "Unknown task question-answering, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

## Ejercicio propuesto: comparación rigurosa de estrategias de generación

### Objetivo
Diseñar un pequeño experimento para comparar cómo cambian las salidas de un LLM causal al modificar los hiperparámetros de generación. No se trata solo de “ver texto”, sino de analizar **diversidad, coherencia, repetición y longitud efectiva**.

### Enunciado
1. Utiliza un mismo *prompt* inicial en español y en inglés.
2. Genera varias respuestas cambiando combinaciones de `temperature`, `top_p` y `max_new_tokens`.
3. Guarda los resultados en una estructura tabular.
4. Calcula métricas sencillas, por ejemplo:
   - longitud en caracteres o tokens,
   - número de palabras únicas,
   - porcentaje de repetición aproximado,
   - aparición de frases truncadas o incoherentes.
5. Redacta una conclusión razonada sobre qué configuración produce mejor equilibrio entre creatividad y estabilidad.

### Qué se espera
Este ejercicio obliga a pasar de la simple ejecución a una **comparación experimental**: mismo prompt, variables controladas, observación de resultados y justificación final. Es una práctica más cercana a un análisis real de comportamiento de modelos.